# NOTEBOOK: 05 — Silver Taxonomy Enrichment

- **Layer:** Silver
- **Purpose:** Enrich Judge results with vendor info, prices and parent category mapping to produce the final enriched Silver table.
- **Inputs:** `main.techmart_silver.taxonomy`, `main.techmart_silver.vendors`, `main.techmart_silver.products`
- **Outputs:** `main.techmart_silver.taxonomy_enriched`
- **Notes:** All records kept regardless of judge status. Category mapping applied here as a business transformation.
- **Author:** Maira Tavares

# 0. Imports and Configuration

In [0]:
from datetime import datetime
from pyspark.sql import functions as F


In [0]:
from utils.config import (
    SILVER_TAXONOMY,
    SILVER_VENDORS,
    SILVER_PRODUCTS,
    TAXONOMY_ENRICHED
)

PROCESSED_AT = datetime.now().isoformat()

In [0]:
print("=" * 55)
print("NOTEBOOK 05 — SILVER TAXONOMY ENRICHMENT")
print("=" * 55)
print(f"  {'Sources':<20} : {SILVER_TAXONOMY}")
print(f"  {'':<20}   {SILVER_VENDORS}")
print(f"  {'':<20}   {SILVER_PRODUCTS}")
print(f"  {'Target':<20} : {TAXONOMY_ENRICHED}")
print(f"  {'Processed at':<20} : {PROCESSED_AT}")
print("=" * 55)
print("✅ Ready to run")

# 1. Load Data

In [0]:
# We read all records from taxonomy regardless of judge_approved status.
# Silver keeps everything for auditability and manual review.

df_taxonomy = spark.table(SILVER_TAXONOMY)
                #    .filter(F.col("judge_approved") == "True")
df_vendors  = spark.table(SILVER_VENDORS)
df_products = spark.table(SILVER_PRODUCTS)

print(f"Approved taxonomy records : {df_taxonomy.count()}")
print(f"Vendors                   : {df_vendors.count()}")
print(f"Products                  : {df_products.count()}")

# 2. Joing tables

In [0]:
# Join and enrich
# Left joins preserve all taxonomy records even if vendor or product data is missing.
# We select only the columns needed downstream — keeping the table lean.

df_enriched = (
    df_taxonomy
    .join(
        df_vendors.select("product_id", "vendor_name"),
        on="product_id",
        how="left"
    )
    .join(
        df_products.select("product_id", "unit_price_usd", "weight_kg"),
        on="product_id",
        how="left"
    )
    .select(
        F.col("product_id").cast("integer"),
        F.col("extracted_name").alias("product_name"),
        F.col("extracted_brand").alias("brand"),
        F.col("extracted_sub_category").alias("sub_category"),
        F.col("vendor_name"),
        F.col("unit_price_usd"),
        F.col("weight_kg"),
        F.col("judge_approved"),
        F.col("judge_taxonomy"),
        F.col("judge_reason"),
        F.col("prompt_version"),
        F.lit(PROCESSED_AT).alias("_processed_at")
    )
)

print(f"Enriched records: {df_enriched.count()}")

# 3. Enrich table mapping categories

In [0]:
# Map sub-categories to parent categories
#
# Mapping logic:
#   sub_category  →  category
#   Televisions   →  Entertainment
#   Computers     →  Computing
#   Phones        →  Mobile Devices
#   Smartwatches  →  Wearables
#   Accessories   →  Accessories
#   Printers      →  Computing
#   Cameras       →  Photography
#   Consoles      →  Entertainment
#   Hardware      →  Computing

category_map = F.create_map(
    F.lit("Televisions"),  F.lit("Entertainment"),
    F.lit("Computers"),    F.lit("Computing"),
    F.lit("Phones"),       F.lit("Mobile Devices"),
    F.lit("Smartwatches"), F.lit("Wearables"),
    F.lit("Accessories"),  F.lit("Accessories"),
    F.lit("Printers"),     F.lit("Computing"),
    F.lit("Cameras"),      F.lit("Photography"),
    F.lit("Consoles"),     F.lit("Entertainment"),
    F.lit("Hardware"),     F.lit("Computing")
)

# coalesce returns "Other" if sub_category is not in the map
df_enriched = df_enriched.withColumn(
    "category",
    F.coalesce(category_map[F.col("sub_category")], F.lit("Other"))
)

print("✅ Category mapping applied")
display(df_enriched.select(
    "product_id", "product_name", "brand",
    "sub_category", "category", "vendor_name",
    "unit_price_usd", "judge_approved"
))

# 4. Save and Validate

In [0]:
# Write enriched taxonomy to Delta
(df_enriched.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TAXONOMY_ENRICHED))

print(f"✅ Written : {TAXONOMY_ENRICHED}")
print(f"   Rows    : {spark.table(TAXONOMY_ENRICHED).count()}")

In [0]:
# Validation summary
approved = spark.table(TAXONOMY_ENRICHED).filter(F.col("judge_approved") == "True").count()
flagged  = spark.table(TAXONOMY_ENRICHED).filter(F.col("judge_approved") == "False").count()
total    = spark.table(TAXONOMY_ENRICHED).count()

print("=" * 55)
print("SILVER TAXONOMY ENRICHMENT — VALIDATION SUMMARY")
print("=" * 55)
print(f"  {'Total records':<20} : {total}")
print(f"  {'Approved':<20} : {approved}")
print(f"  {'Flagged':<20} : {flagged}")
print(f"  {'Target table':<20} : {TAXONOMY_ENRICHED}")
print("=" * 55)
print("✅ Silver taxonomy enrichment complete")

display(spark.table(TAXONOMY_ENRICHED))